# A statistical overview of the Deadlock ranked progression system
<p> This Jupyter Notebook intends on doing a quick tour of the Deadlock ranked progression system, specifically:</p>
<ol>
    <li>Simulating a number of games with a specified winrate and its <strong>predicted point change visualized with a histogram</strong>,</li>
    <li>Simulating a number of games with a range of winrates and seeing the <strong>winrates' affect on the point gain</strong>,</li>
    <li>Training a model to create a function that can (somewhat) accurately <strong>predict rank and point change based on user inputs</strong>.</li>
</ol>

## 1. Getting started with the simulator
<p> This is the main program we'll be using to simulate the ranked system. Please don't skip over this step, <em>unless you already have sufficient data or models!</em></p>
If you don't intend on editing the <em>"dl_RankedSystemSimulator.cpp"</em> then you should only have to click the button below once. </p>

### 1.1 Compile the program
<p><strong> Please do not skip over running the first codeblock as you might encounter issues otherwise!</strong></p>
<p>If you get any errors here, that is normal.<br>  
And if you do get any, please <strong>check the errors logs for more info</strong> so you don't have issues with them later on! </p>

In [ ]:
!g++ dl_RankedSystemSimulator.cpp -o dl_simulator

### 1.2 (Optional) Trying a random simulation
<p> Click the button below to run a <strong>random simulation</strong>, just to see if it works! </p>
<p> Don't worry, if you don't understand much of the output. <br>
However, if you do see any errors, try recompiling it. </p>

In [ ]:
!./dl_simulator

### 1.3 (Optional) Let's try some using some variables
<p> You can change the variables below however you like. I've just put some basic ones to test it out, if you choose not to! <br>
<p> Variables listed out in order: rank, subrank, points in the current subrank, winrate, number of games, delete previous data store, rank adjusted win depreciation, number of simulations. <br>
Explanations and default (safe) values for each are located above them. </p>

In [ ]:
# Your current rank: Initiate, Seeker, Acolyte, Sentinel, Mystic, Ritualist, Emissary, Oracle, Phantom, Ascendant, Eternus
rank = "Emissary"

# Your current subrank: [1-6]
subrank = 1

# Your current points inside the subrank you're in: [0 - 1000] or [0 - 2000]
subrankPoints = 200

# The winrate you project for yourself: [0.00 - 100.00]
# Default: 50.00
winrate = 50

# The number of games you'd like to test the statistics over: [0 - INF]
# Default: 50
games = 50

# Should the program delete it's previous data saves? Usually a good idea to delete these, unless you mean on reusing them long-term [0 = false, 1 = true]:
# Default: 1
deleteCSV = 1

# How much the change in your rank changes your winrate, a small number that tries to account for games getting harder/easier as you rank up/down (small equalizer):
# Default: 0.40 [0.00 - 0.66]
rankWinrateDepreciation = 0.40

# Number of simulation the program will do, usually a large number so we can make a better guess of the statistics (can change depending on your PC's capabilities):
# Default: 10000
simulations = 50000

# Should we show the statistics in the terminal? (May slow down the runtime slightly) [0 = false, 1 = true]:
# Default: 1
terminalOutput = 1

!./dl_simulator {rank} {subrank} {subrankPoints} {winrate} {games} {deleteCSV} {rankWinrateDepreciation} {simulations} {terminalOutput}

## 2. Getting into the statistics and visualizing them
<p> Here we'll install some tools to help us make sense of the simulations we're doing by <strong>visualizing how winrate affects the point gain</strong>. </p>
<p>Another thing we'll do is find the <strong>break-even point</strong>, so the <strong>winrate</strong> at which you <strong>won't gain or lose points</strong> (this is particularly interesting). </p>

## 2.1 Install statistics libraries (and adding some useful functions)
<p> We're installing some tools to help us do some statistical work. 
Make sure you run this and get no errors!</p>

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as scipy
import matplotlib.pyplot as plt

ranks = ("Initiate", "Seeker", "Acolyte", "Sentinel", "Mystic", "Ritualist", "Emissary", "Oracle", "Phantom", "Ascendant", "Eternus")

def get_points(rank, subrank, subrankpts):
    rankNum = ranks.index(rank)
    if(rankNum == -1):
        raise ValueError(f"Rank string is not valid: ({srank})")
    if(subrank < 1 or subrank > 6):
        raise ValueError(f"Subrank value falls outside the range of [1-6]: ({subrank})")
    if(subrankpts < 0 or subrank == 6 and subrankpts >= 2000 or subrank < 6 and subrankPoints >= 1000):
        raise ValueError(f"Ranked points value is invalid: ({subrankpts})")
    subrankNum = subrank - 1
    points = rankNum * 7000 + subrankNum * 1000 + subrankpts
    return points

def get_rank(points):
    if(points > 77000):
        raise ValueError(f"Ranked points value is too high: ({points})")
    if(points < 0): 
        raise ValueError(f"Ranked points are below 0: ({points})")
    _rank = points // 7000
    rank = ranks[_rank]
    subrank = (points % 7000) // 1000
    subrank = subrank if (subrank == 6) else subrank + 1
    subrankPoints = points % 7000
    if (subrank == 6):
        subrankPoints = subrankPoints - 5000
    else:
        subrankPoints %= 1000
    return (rank, subrank, subrankPoints)


### 2.2 Testing out a range of different winrates
<p> Choose to run either of the two codeblocks below with a range of winrates: </p>
<p> 1st - a <strong>random</strong> sample test, so nothing for you to input!<br>
2nd - <strong>fully customizable</strong>, works well if you want to simulate your own rank progression or just want to toy around with the settings! </p>

In [ ]:
## Set rank
# Winrate range
low_bound = 30
high_bound = 70
change = 1 # can be a decimal as well

# Variables
rank = "Mystic"
subrank = 2
points = 0
games = 100
deleteCSV = 1
winRankDepreciation = 0.45
simulations = 10000
terminalOutput = 0

for wr in np.arange(low_bound, high_bound + change, change):
    if wr != low_bound:
        deleteCSV = 0
    !./dl_simulator {rank} {subrank} {points} {wr} {games} {deleteCSV} {winRankDepreciation} {simulations} {terminalOutput}


In [ ]:
## Random rank
# Winrate range
low_bound = 30
high_bound = 70
change = 1 # can be a decimal

# Variables
deleteCSV = 1
terminalOutput = 0

for wr in np.arange(low_bound, high_bound + change, change):
    if wr != low_bound:
        deleteCSV = 0
    !./dl_simulator {wr} {deleteCSV} {terminalOutput}

### 2.3 Taking in the output of the test
<p><strong> Don't skip over this part, if you've clicked either of the two codeblocks above!</strong></p>
<p> You should come back to this each time you decide to rerun either of the codeblocks above.<br>
If you don't you might be stuck with an older test / invalid test. </p>

In [ ]:
output = pd.read_csv("dl_SimulationsOverview.csv")
# Optional test to see if the output works (check that the numbers on the right are equal to (high_bound - low_bound) / change × simulations)
output["winrate"].value_counts() # alternatives: output.describe() or output.info()

### 2.4 Showing the results on a 2D plot
<p> Run either of the two codeblock below and you might just see something interesting! (First shows stats for points, the second one shows stats for rank changes)<br>
If you're confused, this plot is the average number of ranks you'd go up based on the winrate, if you played X amount of games (look at the title of the 2D space). </p>
<p>If you don't see a rough line your test was not done over a range, so refer to the codeblocks in 4.2!</p>

In [ ]:
# Points gained
summary_points = output.groupby('winrate')['totalPointsGained'].agg(['mean', 'std', 'count', 'sem']).reset_index()

fig, ax = plt.subplots(figsize=(9,6))
ax.plot(summary_points['winrate'], summary_points['mean'], marker='o', label='Mean net points gained')
ax.fill_between(summary_points['winrate'], summary_points['mean'] - 1.96*summary_points['sem'], summary_points['mean'] + 1.96*summary_points['sem'], alpha=0.5, label='within 95%')
ax.axhline(0, color='gray', linewidth=1, linestyle=':')
ax.axvline(50, color='gray', linewidth=1, linestyle=':')
ax.set_xlabel('Winrate (%)')
ax.set_ylabel('Average net points gained')
ax.set_title(f"Net points gained vs winrate ({output['successfulGamesPlayed'].mean():.0f} games)")
ax.legend()
plt.show()

In [ ]:
# Rank change
summary_ranks = output.groupby('winrate')['netRankups'].agg(['mean', 'std', 'count', 'sem']).reset_index()

fig, ax = plt.subplots(figsize=(9,6))
ax.plot(summary_ranks['winrate'], summary_ranks['mean'], marker='o', label='Mean net rankups')
ax.fill_between(summary_ranks['winrate'], summary_ranks['mean'] - 1.96*summary_ranks['sem'], summary_ranks['mean'] + 1.96*summary_ranks['sem'], alpha=0.5, label='within 95%')
ax.axhline(0, color='gray', linewidth=1, linestyle=':')
ax.axvline(50, color='gray', linewidth=1, linestyle=':')
ax.set_xlabel('Winrate (%)')
ax.set_ylabel('Average net rank change')
ax.set_title(f"Net rank change vs winrate ({output['successfulGamesPlayed'].mean():.0f} games)")
ax.legend()
plt.show()

### 2.5 Finding the break-even point
<p> I'm guessing you can see the where the line intersects with the 0 on the left (there's a horizontal dotted line), <br>
that's the winrate where you'd stay at the exact same place no matter how many games you played (the break-even point)! <br>
In other words, having a <strong>winrate above the break-even rate means you'll eventually go up in your rank</strong>.</p>

In [ ]:
# Find winrate points where the mean changes sign
breakeven = np.interp(0, summary_points['mean'], summary_points['winrate'])
print(f"Estimated break-even winrate: {breakeven:.2f}%")

### 2.6 (Bonus) Histogram of points gained at a specific winrate
<p> Your break-even winrate is probably below 50%, so what if you had a winrate of 50%? <br>
<strong>On average, how many points would you gain from the amount of games you intended to play?</strong><br>
This is what the codeblock below shows off with a histogram, specifically look at the mean (red line)! </p>
<p> Of course, the winrate is fully customizable, however you would've needed to have computed beforehand, so keep that in mind! <br>
If you get errors here, you probably haven't computed the data for this, I recommend going back to 1.3 <em>(for a specific win rate)</em> or 2.2 <em>(a range of winrates)</em>. 

In [ ]:
# Variable
winrate = 50

output_wr = output[output['winrate'] == winrate]
pointsChange = output_wr['endingPoints'] - output_wr['startingPoints']

plt.hist(pointsChange, bins=50)
plt.axvline(0, color='gray', linestyle=':')
plt.axvline(pointsChange.mean(), color='red', label='Mean')
plt.legend()
plt.title(f"Distribution of points change at {winrate:.2f}% winrate")
plt.show()

## 3. Finding the formula
<p> We've spent a lot of time with the stats now, surely there's a formula then?<br>
Of course! However, this process might take a while, as we'll dynamically construct a model for the formula ourselves.<p>

(However, if you set a rank on either edge of the spectrum, they might be quite inaccurate for you.)</p>
<p>Another note, if you decide to turn on randomness then the formula will be a little bit more inaccurate, but it should still work well as a general purpose model.</p>

### 3.1 Creating a large dataset
<p><strong>You can skip over this, if you have the data already saved on your disk!</strong></p>
<p> This step is quite computationally intensive, especially with larger ranges or a high number of simulations.<br>
(With how the variables are set below, it usually takes me <em>1-2 minutes of runtime</em> - it's quick but maybe slightly inaccurate.) </p>

In [ ]:
# Ranges, lower these if you want a faster (but less accurate) result
range_games = (10, 101, 10) # number of games
range_winrate = (35, 66, 2) # winrate
range_wrd = (0, 1.1, 0.1) # winrate (based on rank) depreciation

# Randomness switch
random = False

# Variables (set these yourself, if you haven't turned random to True)
rank = "Emissary"
subrank = 1
points = 200
simulations = 100 # low, otherwise this is exponentially long

# CSV, don't recommend changing this. Better to start from a clean slate
csvDelete = 1

# Creating data for our model
for _games in np.arange(*range_games):
    for _winrate in np.arange(*range_winrate):
        for _wdr in np.arange(*range_wrd):
            if(random):
                !./dl_simulator {"Random"} {1} {0} {_winrate} {_games} {csvDelete} {_wdr} {simulations} {0}
            else:
                !./dl_simulator {rank} {subrank} {points} {_winrate} {_games} {csvDelete} {_wdr} {simulations} {0}
            csvDelete = 0


### 3.2 Loading the dataset and training the model
<p><strong>You can skip over this, if you have the model already on your disk!</strong></p> 
<p> We're going to load the data and train our model with it. <br>
If you want to see the output of the model and it's coefficients in the raw form, just flip the outputBool to True!<br>
Shouldn't take more than a second.</p>

In [ ]:
# Read data
model_data = pd.read_csv("dl_SimulationsOverview.csv")

# Group the data
model_data.groupby(['successfulGamesPlayed', 'winrate', 'wdr']).size()

# Create models
outputBool = False

import statsmodels.formula.api as smf
model_points = smf.ols('totalPointsGained ~ successfulGamesPlayed * winrate + wdr', data=model_data).fit()
if(outputBool):
    print(model_points.summary())
model_ranks = smf.ols('netRankups ~ successfulGamesPlayed * winrate', data=model_data).fit()
if(outputBool):
    print(model_rankups.summary())

### 3.3 Loading or saving a model
<p> I recommend saving your models in case you want to use them again later on, you can come back to here and load them again. </br>
For reference though, training a model isn't that time consuming, but it saves you having to have the CSV file on hand (which is significantly bigger than the model) </p>

In [ ]:
# Loading models from files (if you have them)
import statsmodels.api as sm
model_points = sm.load("model_points.pickle")
model_ranks = sm.load("model_ranks.pickle")

In [ ]:
# Saving models to files (if you want to save them)
model_points.save("model_points.pickle")
model_ranks.save("model_ranks.pickle")

### 3.4 (Optional) Test whether the model is well-aligned
<p> You should expect a rough upwards line here that doesn't stray far away from the starting point of 0, this is mostly just for a sanity check on the formula. <br>
If it is broken in any way, I recommend retraining the model or recalculating the data and then retraining the model. </p> 

In [ ]:
# Variables
range_games = (1, 102, 5)
winrate = 45
wdr = 0.45 # don't change this outside [0.00 - 0.50]
model = model_points

games_values = list(range(*range_games))
results = []
for _games in games_values:
    results.append(predict_stat_gained(_games, winrate, wdr, model))

fig, ax = plt.subplots(figsize=(9,6))
ax.plot(games_values, results, marker='o', label='Average points')
ax.set_xticks(games_values)
ax.set_xlabel('Number of games')
ax.set_ylabel('Average net points change')
ax.set_title(f"Net points change in ({games_values[0]} - {games_values[-1]} games)")
ax.legend()
plt.show()

### 3.5 Checking the formula and trying it out
<p> Below we have 2 codeblocks: <br>
1. Gets the formula coefficients and showcases the formula and its accuracy <br>
2. Tests out the formula and gives you a way to predict the statistics based on your inputs. You can also predict your rank, if you input your stats </p>

<p><strong> Important: </strong> you will likely see a disparity in the <em>"average rank change"</em> number and the actual and more accurate <em>"predicted rank change"</em>, this is because the former does not accurately account for differences between 1000 and 2000 point large subrank brackets unfortunately, as it only uses averages. It is better to use the point formula to accurately predict your rank, if you want to do that by hand</p>

In [ ]:
# Function to get coefficients for the formula
def get_coef(model, roundTo=0):
    values = model.params.values.copy()
    size = values.size

    rounded_values = values
    if(roundTo == 0):
        rounded_values[size-1] = round(values[size-1] * 100)
        rounded_values = rounded_values.astype(int)
    else:
        rounded_values = rounded_values.round(roundTo)
    return rounded_values

# Coefficients & formulas
coefP = get_coef(model_points)
coefR = get_coef(model_ranks, roundTo=3)
r_squaredP = round(model_points.rsquared * 1000) / 10
r_squaredR = round(model_ranks.rsquared * 1000) / 10

print("The models are able to account for " + str(r_squaredP) + "% (points change) and " + str(r_squaredR) + "% (rank change)")
print("Formula for points change: " + str(coefP[0]) + " - " + str(-coefP[1]) + " × (games) + " + str(coefP[2]) + " × (winrate) + " + str(coefP[3]) + " × (games) × (winrate)" +  " - " + str(-coefP[4]) + " × (win rank depreciation)")
print("Formula for rank change: " + str(coefR[0]) + " - " + str(-coefR[1]) + " × (games) + " + str(coefR[2]) + " × (winrate)")

In [ ]:
# Check and test out the formulas
games = 100
winrate = 50
wdr = 0.40 # don't change this outside [0.00 - 0.60]

# If you're curious how this would affect your rank, toggle the switch below
rankPredict = True

# If you decided to turn the switch above on, then set these yourself
rank = "Emissary"
subrank = 3
subrankPoints = 0

# Function to predict stats based on a model
def predict_stat_gained(games, winrate, wdr, model):
    input_df = pd.DataFrame({
            'successfulGamesPlayed': [games],
            'winrate': [winrate],
            'wdr': [wdr*100]
    })
    return model.predict(input_df).iloc[0]

# Results
points_gained = round(predict_stat_gained(games, winrate, wdr, model_points), 2)
ranks_gained = round(predict_stat_gained(games, winrate, wdr, model_ranks), 2)

if(rankPredict):
    currentPoints = get_points(rank, subrank, subrankPoints)
    predictedPoints = currentPoints + int(round(points_gained, 0))
    predictedValues = get_rank(predictedPoints)
    currentFloor = 2000 if (subrank == 6) else 1000
    predictedFloor = 2000 if (predictedValues[1] == 6) else 1000
    print(f"Predicted rank change: {rank} {subrank} ({subrankPoints}/{currentFloor}) -> {predictedValues[0]} {predictedValues[1]} ({predictedValues[2] % predictedFloor}/{predictedFloor})")
    
print("Average points change: " + str(points_gained))
print("Average rank change: " + str(ranks_gained))